# 🔬 Notebook 1: Segmentation Comparison — CNN vs RNN vs Transformer
Train 3 custom segmentation models on 80×80 RBC images + binary masks.
**Metrics:** Dice Coefficient, Mean IoU

In [13]:
# ── 0. Install dependencies (run once on Colab) ──────────────────────────────
# !pip install -q torchmetrics scikit-learn matplotlib seaborn tqdm pillow


In [14]:
# ── 1. Imports & GPU check ───────────────────────────────────────────────────
import os, random, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
torch.manual_seed(42)


Device: cuda


In [15]:
# ── 2. Config ────────────────────────────────────────────────────────────────
# ▶ LOCAL path (dùng khi connect Colab Runtime đến local kernel)
DATASET_BASE = "/content/Dataset" 

# ▶ COLAB path (uncomment nếu upload data lên Google Drive)
# from google.colab import drive; drive.mount('/content/drive')
# DATASET_BASE = "/content/drive/MyDrive/Project/Dataset"

SLIDE_DIRS = [
    "Elsafty_RBCs_for_Segmentation_and_Detection_Slide_2",
    "Elsafty_RBCs_for_Segmentation_and_Detection_Slide_3",
]

IMG_SIZE   = 80
BATCH_SIZE = 32
EPOCHS     = 50
LR         = 1e-3
MAX_SAMPLES = 5000   # tổng ảnh dùng để train (giảm nếu Colab hết RAM)

RESULTS_DIR = os.path.join(os.path.dirname(DATASET_BASE), "results", "segmentation")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Config OK")


Config OK


In [16]:
# ── 3. Dataset ────────────────────────────────────────────────────────────────
import zipfile

class RBCSegDataset(Dataset):
    def __init__(self, image_paths, mask_paths, img_size=80):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.img_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),
        ])
        self.mask_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
        ])

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        img  = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")
        return self.img_tf(img), (self.mask_tf(mask) > 0.5).float()


def collect_paths(base, slide_dirs, max_samples):
    imgs, masks = [], []
    for sd in slide_dirs:
        sd_path = Path(base) / sd
        print(f"\n[INFO] Đang quét kiểm tra thư mục: {sd_path}")
        if not sd_path.exists():
            print(f"⚠ LỖI: Thư mục {sd_path} hoàn toàn không tồn tại trên ổ cứng!")
            continue
            
        # 1. Tự động kiểm tra file ảnh (nhanh, dùng next() thay vì load nguyên list gây quá tải RAM)
        has_imgs = next(sd_path.rglob("*.png"), None) is not None or next(sd_path.rglob("*.jpg"), None) is not None
        
        if not has_imgs:
            print(f"Tiến hành giải nén tự động cho: {sd} ...")
            zips = list(sd_path.glob("*.zip"))
            if not zips:
                print(f"⚠ Bỏ qua {sd_path}: Thư mục không có ảnh và không có file .zip!")
                continue
            for zip_path in zips:
                print(f"  Đang giải nén: {zip_path.name}...")
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(sd_path)
                    
        # 2. Phân loại ảnh/mask nhanh gọn
        print(f"[INFO] Bắt đầu duyệt ảnh CROP và MASK trong thư mục: {sd} ...")
        img_files, mask_files = [], []
        allowed_exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif"}
        
        for p in sd_path.rglob("*.*"):
            if not p.is_file() or p.name.startswith("."): 
                continue
            if p.suffix.lower() not in allowed_exts:
                continue
                
            path_str = str(p.parent).lower()
            if "mask" in path_str or "label" in path_str:
                mask_files.append(p)
            elif "crop" in path_str or "image" in path_str:
                img_files.append(p)

        print(f"👉 Kết quả phân loại thư mục {sd}: Tìm thấy {len(img_files)} files Crop và {len(mask_files)} files Mask.")
        
        if not img_files or not mask_files:
            print(f"⚠ Bỏ qua {sd}: Không có đủ ảnh/mask để ghép.")
            continue
            
        # 3. Ghép cặp ảnh-mask
        mask_dict = {m.stem: str(m) for m in mask_files}
        matched = 0
        for img_p in img_files:
            stem = img_p.stem
            if stem in mask_dict:
                imgs.append(str(img_p))
                masks.append(mask_dict[stem])
                matched += 1
                
        if matched == 0:
            print(f"⚠ Cảnh báo {sd}: Không có cặp nào khớp tên với nhau!")
            
    if len(imgs) == 0:
        raise ValueError("Data rỗng! Không có cặp ảnh/mask nào được phân loại thành công. Vui lòng đọc kĩ log [INFO] và [⚠ Cảnh báo] in ra phía trên để tìm nguyên nhân.")
        
    idx = random.sample(range(len(imgs)), min(max_samples, len(imgs)))
    return [imgs[i] for i in idx], [masks[i] for i in idx]


# ──────── Tiến hành setup Dataset & Dataloader ────────
print("--- [BẮT ĐẦU QUÁ TRÌNH TẠO DATASET] ---")
img_paths, mask_paths = collect_paths(DATASET_BASE, SLIDE_DIRS, MAX_SAMPLES)
print(f"👍 Total samples after pairing: {len(img_paths)}")

dataset = RBCSegDataset(img_paths, mask_paths, IMG_SIZE)
print(f"Dataset size: {len(dataset)}")

# Chia train/val/test = 70/15/15
n_train = int(0.7 * len(dataset))
n_val   = int(0.15 * len(dataset))
n_test  = len(dataset) - n_train - n_val
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test])

# Lưu ý: num_workers=0 để không đụng độ multiprocessing trên Windows
train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_dl   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_dl  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train/Val/Test phân bổ: {n_train}/{n_val}/{n_test}")


--- [BẮT ĐẦU QUÁ TRÌNH TẠO DATASET] ---

[INFO] Đang quét kiểm tra thư mục: /content/Dataset/Elsafty_RBCs_for_Segmentation_and_Detection_Slide_2
⚠ LỖI: Thư mục /content/Dataset/Elsafty_RBCs_for_Segmentation_and_Detection_Slide_2 hoàn toàn không tồn tại trên ổ cứng!

[INFO] Đang quét kiểm tra thư mục: /content/Dataset/Elsafty_RBCs_for_Segmentation_and_Detection_Slide_3
⚠ LỖI: Thư mục /content/Dataset/Elsafty_RBCs_for_Segmentation_and_Detection_Slide_3 hoàn toàn không tồn tại trên ổ cứng!


ValueError: Data rỗng! Không có cặp ảnh/mask nào được phân loại thành công. Vui lòng đọc kĩ log [INFO] và [⚠ Cảnh báo] in ra phía trên để tìm nguyên nhân.

## 🏗 Model Architectures
### Pipeline A — Custom CNN (Mini U-Net)

In [ ]:
# ── 4A. CNN — Mini U-Net ─────────────────────────────────────────────────────
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class MiniUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, features=[16, 32, 64]):
        super().__init__()
        self.encoders  = nn.ModuleList()
        self.pools     = nn.ModuleList()
        self.decoders  = nn.ModuleList()
        self.upconvs   = nn.ModuleList()

        prev = in_ch
        for f in features:
            self.encoders.append(DoubleConv(prev, f))
            self.pools.append(nn.MaxPool2d(2))
            prev = f

        self.bottleneck = DoubleConv(prev, prev * 2)
        prev = prev * 2

        for f in reversed(features):
            self.upconvs.append(nn.ConvTranspose2d(prev, f, 2, 2))
            self.decoders.append(DoubleConv(f * 2, f))
            prev = f

        self.head = nn.Conv2d(prev, out_ch, 1)

    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x); skips.append(x); x = pool(x)
        x = self.bottleneck(x)
        skips = skips[::-1]
        for up, dec, skip in zip(self.upconvs, self.decoders, skips):
            x = up(x)
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])
            x = dec(torch.cat([skip, x], dim=1))
        return self.head(x)

model_cnn = MiniUNet().to(DEVICE)
params_cnn = sum(p.numel() for p in model_cnn.parameters())
print(f"CNN U-Net params: {params_cnn:,}")


CNN U-Net params: 482,737


### Pipeline B — Custom RNN (ConvLSTM U-Net)

In [ ]:
# ── 4B. RNN — ConvLSTM U-Net ─────────────────────────────────────────────────
class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hidden_ch, kernel_size=3):
        super().__init__()
        self.hidden_ch = hidden_ch
        pad = kernel_size // 2
        self.gates = nn.Conv2d(in_ch + hidden_ch, 4 * hidden_ch, kernel_size, padding=pad)

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        i, f, g, o = self.gates(combined).chunk(4, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c + i * g
        h = o * torch.tanh(c)
        return h, c

    def init_hidden(self, B, H, W, device):
        return (torch.zeros(B, self.hidden_ch, H, W, device=device),
                torch.zeros(B, self.hidden_ch, H, W, device=device))


class ConvLSTMUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, features=[16, 32, 64]):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.pools    = nn.ModuleList()
        self.decoders = nn.ModuleList()
        self.upconvs  = nn.ModuleList()

        prev = in_ch
        for f in features:
            self.encoders.append(DoubleConv(prev, f))
            self.pools.append(nn.MaxPool2d(2))
            prev = f

        bn_ch = prev * 2
        self.bn_conv     = nn.Conv2d(prev, bn_ch, 1)
        self.conv_lstm   = ConvLSTMCell(bn_ch, bn_ch)

        prev = bn_ch
        for f in reversed(features):
            self.upconvs.append(nn.ConvTranspose2d(prev, f, 2, 2))
            self.decoders.append(DoubleConv(f * 2, f))
            prev = f

        self.head = nn.Conv2d(prev, out_ch, 1)

    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x); skips.append(x); x = pool(x)

        x  = self.bn_conv(x)
        B, C, H, W = x.shape
        h, c = self.conv_lstm.init_hidden(B, H, W, x.device)
        # treat spatial rows as time steps
        for t in range(H):
            row = x[:, :, t:t+1, :].expand(-1, -1, H, W)
            h, c = self.conv_lstm(row, h, c)
        x = h

        skips = skips[::-1]
        for up, dec, skip in zip(self.upconvs, self.decoders, skips):
            x = up(x)
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])
            x = dec(torch.cat([skip, x], dim=1))
        return self.head(x)

model_rnn = ConvLSTMUNet().to(DEVICE)
params_rnn = sum(p.numel() for p in model_rnn.parameters())
print(f"ConvLSTM U-Net params: {params_rnn:,}")


ConvLSTM U-Net params: 1,449,521


### Pipeline C — Custom Transformer (ViT-UNet)

In [ ]:
# ── 4C. Transformer — ViT-UNet ───────────────────────────────────────────────
class PatchEmbed(nn.Module):
    def __init__(self, img_size=80, patch_size=8, in_ch=3, embed_dim=64):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches  = (img_size // patch_size) ** 2
        self.h_patches  = img_size // patch_size
        self.proj = nn.Conv2d(in_ch, embed_dim, patch_size, patch_size)

    def forward(self, x):
        x = self.proj(x)                          # B,E,H/P,W/P
        B, E, H, W = x.shape
        tokens = x.flatten(2).transpose(1, 2)     # B, N, E
        return tokens, H, W


class TransBlock(nn.Module):
    def __init__(self, dim=64, heads=4, mlp_ratio=3, drop=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, heads, dropout=drop, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio), nn.GELU(), nn.Dropout(drop),
            nn.Linear(dim * mlp_ratio, dim), nn.Dropout(drop))

    def forward(self, x):
        n = self.norm1(x)
        x = x + self.attn(n, n, n)[0]
        x = x + self.mlp(self.norm2(x))
        return x


class ViTUNet(nn.Module):
    def __init__(self, img_size=80, patch_size=8, in_ch=3, out_ch=1,
                 embed_dim=64, depth=4, heads=4):
        super().__init__()
        self.patch = PatchEmbed(img_size, patch_size, in_ch, embed_dim)
        n = self.patch.n_patches
        self.pos_embed = nn.Parameter(torch.randn(1, n, embed_dim) * 0.02)
        self.blocks    = nn.Sequential(*[TransBlock(embed_dim, heads) for _ in range(depth)])
        self.norm      = nn.LayerNorm(embed_dim)
        # CNN decoder to restore spatial resolution
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, 64, 2, 2),  # ×2
            nn.ReLU(), nn.BatchNorm2d(64),
            nn.ConvTranspose2d(64, 32, 2, 2),         # ×2
            nn.ReLU(), nn.BatchNorm2d(32),
            nn.ConvTranspose2d(32, 16, 2, 2),         # ×2 → 80×80 for patch=8
            nn.ReLU(), nn.BatchNorm2d(16),
            nn.Conv2d(16, out_ch, 1),
        )

    def forward(self, x):
        tokens, H, W = self.patch(x)
        tokens = tokens + self.pos_embed
        tokens = self.blocks(tokens)
        tokens = self.norm(tokens)                    # B,N,E
        B, N, E = tokens.shape
        spatial = tokens.transpose(1, 2).reshape(B, E, H, W)
        return self.decoder(spatial)

model_vit = ViTUNet().to(DEVICE)
params_vit = sum(p.numel() for p in model_vit.parameters())
print(f"ViT-UNet params: {params_vit:,}")

print(f"\nParameter Summary:")
print(f"  CNN  U-Net : {params_cnn:>8,}")
print(f"  RNN  U-Net : {params_rnn:>8,}")
print(f"  ViT  U-Net : {params_vit:>8,}")


ViT-UNet params: 212,769

Parameter Summary:
  CNN  U-Net :  482,737
  RNN  U-Net : 1,449,521
  ViT  U-Net :  212,769


## 🏋 Training

In [ ]:
# ── 5. Loss & Metrics ────────────────────────────────────────────────────────
def dice_loss(pred, target, eps=1e-6):
    pred   = torch.sigmoid(pred)
    num    = 2 * (pred * target).sum(dim=(1,2,3))
    den    = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) + eps
    return (1 - num / den).mean()

def bce_dice_loss(pred, target):
    return nn.BCEWithLogitsLoss()(pred, target) + dice_loss(pred, target)

def dice_score(pred, target, eps=1e-6):
    pred   = (torch.sigmoid(pred) > 0.5).float()
    num    = 2 * (pred * target).sum()
    den    = pred.sum() + target.sum() + eps
    return (num / den).item()

def iou_score(pred, target, eps=1e-6):
    pred   = (torch.sigmoid(pred) > 0.5).float()
    inter  = (pred * target).sum()
    union  = pred.sum() + target.sum() - inter + eps
    return (inter / union).item()


In [ ]:
# ── 6. Training loop ─────────────────────────────────────────────────────────
def train_model(model, name, epochs=EPOCHS, lr=LR):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    # ReduceLR theo mode 'max' vì ta giám sát Dice Score
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

    early_stop_patience = 10
    epochs_no_improve = 0

    history = {"train_loss": [], "val_dice": [], "val_iou": []}
    best_dice, best_state = 0.0, None

    for epoch in range(1, epochs + 1):
        # ── train ──
        model.train()
        total_loss = 0
        for imgs, masks in train_dl:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            loss = bce_dice_loss(model(imgs), masks)
            loss.backward()
            # Kỹ thuật: Gradient Clipping giúp tránh exploding gradients (đặc biệt cho RNN)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_dl)

        # ── validate ──
        model.eval()
        val_dice_sum, val_iou_sum = 0., 0.
        with torch.no_grad():
            for imgs, masks in val_dl:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                preds = model(imgs)
                val_dice_sum += dice_score(preds, masks)
                val_iou_sum  += iou_score(preds, masks)

        avg_dice = val_dice_sum / len(val_dl)
        avg_iou  = val_iou_sum  / len(val_dl)
        scheduler.step(avg_dice)

        history["train_loss"].append(avg_loss)
        history["val_dice"].append(avg_dice)
        history["val_iou"].append(avg_iou)

        if avg_dice > best_dice:
            best_dice  = avg_dice
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epoch % 5 == 0 or epoch == 1:
            curr_lr = optimizer.param_groups[0]['lr']
            print(f"[{name}] Epoch {epoch:3d}/{epochs} | LR {curr_lr:.1e} | "
                  f"Loss {avg_loss:.4f} | Dice {avg_dice:.4f} | IoU {avg_iou:.4f}")
        
        # Kỹ thuật: Early Stopping
        if epochs_no_improve >= early_stop_patience:
            print(f"🛑 Early stopping tại epoch {epoch} (Không cải thiện sau {early_stop_patience} epochs)")
            break

    # load best
    model.load_state_dict(best_state)
    torch.save(best_state, f"{RESULTS_DIR}/{name}_best.pt")
    print(f"✅ {name} best Dice = {best_dice:.4f}")
    return history


In [ ]:
# ── 7. Train all 3 models ────────────────────────────────────────────────────
t0 = time.time()
hist_cnn = train_model(model_cnn, "CNN_UNet")
print(f"⏱ CNN training time: {(time.time()-t0)/60:.1f} min\n")

t0 = time.time()
hist_rnn = train_model(model_rnn, "RNN_UNet")
print(f"⏱ RNN training time: {(time.time()-t0)/60:.1f} min\n")

t0 = time.time()
hist_vit = train_model(model_vit, "ViT_UNet")
print(f"⏱ ViT training time: {(time.time()-t0)/60:.1f} min")


c:\ProgramData\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## 📊 Evaluation & Comparison

In [ ]:
# ── 8. Test evaluation ───────────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    dice_sum, iou_sum = 0., 0.
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            dice_sum += dice_score(preds, masks)
            iou_sum  += iou_score(preds, masks)
    n = len(loader)
    return dice_sum / n, iou_sum / n

results = {}
for m, name in [(model_cnn, "CNN U-Net"), (model_rnn, "RNN U-Net"), (model_vit, "ViT-UNet")]:
    d, i = evaluate(m, test_dl)
    results[name] = {"Dice": d, "IoU": i}
    print(f"{name:15s} → Dice: {d:.4f}  IoU: {i:.4f}")


In [ ]:
# ── 9. Plot training curves & comparison ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for hist, label, color in [
    (hist_cnn, "CNN U-Net",  "steelblue"),
    (hist_rnn, "RNN U-Net",  "coral"),
    (hist_vit, "ViT-UNet",   "seagreen"),
]:
    axes[0].plot(hist["train_loss"], label=label, color=color)
    axes[1].plot(hist["val_dice"],   label=label, color=color)
    axes[2].plot(hist["val_iou"],    label=label, color=color)

for ax, title in zip(axes, ["Train Loss", "Val Dice", "Val IoU"]):
    ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle("Segmentation: CNN vs RNN vs Transformer", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/training_curves.png", dpi=150)
plt.show()

# Bar chart comparison
names  = list(results.keys())
dices  = [results[n]["Dice"] for n in names]
ious   = [results[n]["IoU"]  for n in names]
x      = np.arange(len(names))
width  = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, dices, width, label="Dice", color=["steelblue","coral","seagreen"])
ax.bar(x + width/2, ious,  width, label="IoU",  color=["steelblue","coral","seagreen"], alpha=0.6)
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylim(0, 1); ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_title("Test Set: Dice & IoU Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/metric_comparison.png", dpi=150)
plt.show()


In [ ]:
# ── 10. Visual prediction samples ────────────────────────────────────────────
def show_predictions(models_names, loader, n=3):
    imgs_b, masks_b = next(iter(loader))
    imgs_b, masks_b = imgs_b[:n].to(DEVICE), masks_b[:n].to(DEVICE)

    fig, axes = plt.subplots(n, 2 + len(models_names), figsize=(14, n * 3.5))
    for i in range(n):
        img_np  = imgs_b[i].cpu().permute(1,2,0).numpy()
        img_np  = (img_np * 0.5 + 0.5).clip(0, 1)
        mask_np = masks_b[i].cpu().squeeze().numpy()

        axes[i][0].imshow(img_np);   axes[i][0].set_title("Input")
        axes[i][1].imshow(mask_np, cmap="gray"); axes[i][1].set_title("GT Mask")

        for j, (model, name) in enumerate(models_names):
            with torch.no_grad():
                pred = torch.sigmoid(model(imgs_b[i:i+1])).squeeze().cpu().numpy()
            axes[i][2+j].imshow(pred > 0.5, cmap="gray")
            axes[i][2+j].set_title(name)

        for ax in axes[i]: ax.axis("off")

    plt.suptitle("Segmentation Predictions", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/prediction_samples.png", dpi=150)
    plt.show()

show_predictions(
    [(model_cnn, "CNN"), (model_rnn, "RNN"), (model_vit, "ViT")],
    test_dl
)
